# Building a Semantic Graph

So far I have been looking at individual complaint retrieval.

Starting from a complaint, I can retrieve the most similar complaints using embeddings and cosine similarity.

The next question is more interesting:

> Can local similarity relationships reveal larger complaint communities?

To explore this, I will build a graph where:

- each complaint is a node
- semantic similarity creates edges
- communities emerge from local relationships

The ultimate goal is to move from individual complaints to recurring patterns and investigation workflows.

In [2]:
import sys
from pathlib import Path

sys.path.append(
    str(Path("..").resolve())
)

import pandas as pd
import networkx as nx

from src.embeddings import (
    load_model,
    generate_embeddings
)

from src.retrieval import (
    get_neighbors
)

## Load data

I use the same complaint sample used in the retrieval notebook.

Keeping the dataset fixed makes it easier to compare results across experiments.

In [3]:
df_sample = pd.read_parquet(
    "../data/processed/complaints_50k.parquet"
)

texts = (
    df_sample["Consumer complaint narrative"]
    .dropna()
    .astype(str)
    .tolist()
)

sample_texts = texts[:2000]

print(
    f"{len(sample_texts):,} complaints loaded."
)

2,000 complaints loaded.


## Generate embeddings

The graph will be built entirely from semantic relationships between complaint narratives.

In [4]:
model = load_model()

embeddings = generate_embeddings(
    sample_texts,
    model
)

embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

(2000, 384)

### Build semantic relationships

For each complaint, I retrieve its nearest semantic neighbors.

Each neighbor relationship becomes an edge in the graph.

If the embeddings capture meaningful structure, complaints discussing similar issues should become connected through the network.

In [40]:
k = 5
threshold = 0.72

edges = []

for i in range(len(embeddings)):

    neighbors, similarities = get_neighbors(
        embeddings,
        query_index=i,
        k=k + 1
    )

    for neighbor in neighbors:

        if neighbor == i:
            continue

        similarity = similarities[neighbor]

        if similarity < threshold:
            continue

        edges.append(
            (
                i,
                neighbor,
                similarity
            )
        )

print(f"{len(edges):,} edges created")

4,107 edges created


In [41]:
edges[:5]

[(1, 1718, 1.0),
 (1, 460, 0.7789153),
 (1, 1138, 0.7712364),
 (1, 163, 0.76004714),
 (1, 291, 0.757929)]

### Build the graph

In [43]:
G = nx.Graph()

for source, target, similarity in edges:

    G.add_edge(
        source,
        target,
        weight=similarity
    )


print(
    f"Nodes: {G.number_of_nodes():,}"
)

print(
    f"Edges: {G.number_of_edges():,}"
)

Nodes: 1,075
Edges: 2,939


In [45]:
query_index = 1

neighbors = list(
    G.neighbors(query_index)
)

len(neighbors)

5

In [47]:
# Let's see the neighbors of the first complaint
query_index = 1

neighbors = list(
    G.neighbors(query_index)
)

neighbors

[1718, 460, 1138, 163, 291]

In [48]:
# Let's see the size of the community around the first complaint

community = nx.node_connected_component(
    G,
    query_index
)

len(community)

941

In [49]:
all_similarities = [edge[2] for edge in edges]

pd.Series(all_similarities).describe()

count    4107.000000
mean        0.821809
std         0.094156
min         0.720018
25%         0.747163
50%         0.780721
75%         0.900032
max         1.000000
dtype: float64

In [51]:
print(G.number_of_nodes())
print(G.number_of_edges())

community = nx.node_connected_component(
    G,
    1
)

print(len(community))

1075
2939
941


In [53]:
import umap.umap_ as umap

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

embedding_2d = reducer.fit_transform(
    embeddings
)

c:\Users\pablo\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [57]:
import pandas as pd
import plotly.express as px
import networkx as nx

query_index = 1

component = nx.node_connected_component(
    G,
    query_index
)

print(f"Community size: {len(component)}")

plot_df = pd.DataFrame({
    "x": embedding_2d[:, 0],
    "y": embedding_2d[:, 1]
})

plot_df["group"] = "Other"

plot_df.loc[list(component), "group"] = "Community"
plot_df.loc[query_index, "group"] = "Query"

plot_df["size"] = 3
plot_df.loc[list(component), "size"] = 5
plot_df.loc[query_index, "size"] = 25

fig = px.scatter(
    plot_df,
    x="x",
    y="y",
    color="group",
    size="size",
    opacity=0.7,
    hover_data=[plot_df.index]
)

fig.update_layout(
    width=1000,
    height=800,
    title=f"Connected component around complaint {query_index}"
)

fig.show()

Community size: 941


### Visual inspection

The connected component associated with complaint 1 occupies a large and coherent region of the embedding space.

However, it also extends into neighboring regions through chains of semantically related complaints.

This suggests that connected components capture broad semantic territories rather than well-defined complaint communities.

A more suitable approach may be to identify dense subgroups within the connected component rather than treating the entire component as a single community.